# Week 2, day 4 (afternoon) — Worksheet 08 SOLUTIONS: the structures at work

Every cell below was executed on the same Python the lab ships; the quoted
output is real, including the error in Q2.

Q2 raises on purpose, so running straight through stops there. Run the cells
after it individually.

Read Q11 even if you skip it. Several numbers on this sheet are correct and
misleading at the same time, and Q11 is where that gets named.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 08 — The structures at work. Run this once.
from array import array

# One dict per record, exactly as a JSON feed would hand it to you.
# This data is deliberately dirty -- see the intro.
raw_events = [
    {"event_id": "e1", "user": "u01", "action": "view",     "amount": None,    "ts": "2026-08-20"},
    {"event_id": "e2", "user": "u02", "action": "purchase", "amount": "49.99", "ts": "2026-08-20"},
    {"event_id": "e3", "user": "u01", "action": "purchase", "amount": "12.50", "ts": "2026-08-21"},
    {"event_id": "e2", "user": "u02", "action": "purchase", "amount": "49.99", "ts": "2026-08-20"},
    {"event_id": "e4", "user": "u03", "action": "view",     "amount": None,    "ts": "2026-08-21"},
    {"event_id": "e5", "user": "u99", "action": "purchase", "amount": "5.00",  "ts": "2026-08-22"},
    {"event_id": "e6", "user": "u01", "action": "refund",   "amount": "12.50", "ts": "2026-08-22"},
    {"event_id": "e7", "user": "u02", "action": "purchase",                    "ts": "2026-08-23"},
]

# The dimension table you join against. Note who is NOT in it.
users = {
    "u01": {"name": "Ada", "region": "East"},
    "u02": {"name": "Bo",  "region": "West"},
    "u03": {"name": "Cai", "region": "East"},
}

# The contract every record is supposed to satisfy.
REQUIRED_FIELDS = {"event_id", "user", "action", "amount", "ts"}

print("raw records:", len(raw_events))
print("known users:", len(users))

PART A — Picking the right structure

The same column, four ways. Each answers a different question, and picking
wrong is how a report ends up quietly meaning something else.

### Question 1

Warm-up — one column, three structures. -> the list is 8 long with repeats; the set is `{u01, u02, u03, u99}`, length `4`; the counts are `{'u01': 3, 'u02': 3, 'u03': 1, 'u99': 1}`.

THE WHOLE SHEET IN ONE QUESTION. Same column, three answers: `8`, `4`, and
a number per user. None is more correct than the others; they answer
different questions, and the failure mode is answering the wrong one
confidently.

"How many users" is the SET. Reporting the list length instead — 8 — would
double your user base, and nothing in the output would look wrong.

In [ ]:
user_list = [event["user"] for event in raw_events]
print(user_list, len(user_list))

user_set = set(user_list)
print(user_set, len(user_set))

user_counts = {}
for user in user_list:
    if user not in user_counts:
        user_counts[user] = 0
    user_counts[user] = user_counts[user] + 1
print(user_counts)

# "How many users" -> the SET (len 4). The list would answer "how many
# events", which is a different and larger number.
# "Busiest user" -> the DICT, because only it keeps a count per user.

### Question 2

list vs array. -> `array('d', [49.99, 12.5, 5.0])`, length `3`, typecode `d`; after append, `array('d', [49.99, 12.5, 5.0, 1.25])`; then `TypeError: must be real number, not str`.

An array holds ONE type and enforces it at the moment of insertion. A list
would have accepted `"free"` silently and blown up later inside `sum()` —
far from the line that caused it.

That trade is the whole point: a list is flexible, an array is strict and
compact. Note `12.50` came back as `12.5` — stored as a float64, not as the
text you typed.

WHERE THIS GOES IN REAL WORK: `array` is rarely used directly. It is the
idea underneath numpy and pandas — one type per column, stored compactly,
operated on as a whole. Neither is installed here; when you meet them,
recognise this as the same bargain scaled up.

In [ ]:
amounts = array("d", [49.99, 12.50, 5.00])   # "d" = double, i.e. float64
print(amounts, len(amounts), amounts.typecode)

amounts.append(1.25)
print(amounts)

# This is SUPPOSED to raise. An array enforces its element type; a list
# would have accepted "free" without a word and broken your sum() later.
amounts.append("free")

PART B — Ingestion: cleaning the feed

### Question 3

Deduplicating the feed. -> `in: 8 kept: 7 dropped: ['e2']`.

The `seen` set does the remembering and the `clean` list keeps the order —
neither structure could do the job alone. A bare `set(raw_events)` would
not even run: dictionaries are unhashable, so they cannot go in a set.

This is the standard first stage of any ingestion pipeline, and note it
reports what it dropped. A dedup step that silently discards rows is
indistinguishable from a dedup step with a bug in it.

In [ ]:
clean = []
seen = set()
duplicates = []
for event in raw_events:
    key = event["event_id"]
    if key in seen:               # membership on a set: the fast lookup
        duplicates.append(key)
        continue
    seen.add(key)
    clean.append(event)

print("in:", len(raw_events), "kept:", len(clean), "dropped:", duplicates)

### Question 4

Schema validation. -> `e1`–`e6` each print `set()`, `e7` prints `{'amount'}`, and `failed schema: ['e7']`.

`REQUIRED_FIELDS - set(event)` is the entire check. `set(a_dict)` gives its
keys, and the difference is exactly the fields that should be there and are
not — no loop over field names, no `if` per field.

An empty `set()` printing for the good records is the honest output: the
check ran and found nothing, which is different from the check not running.

In [ ]:
failed = []
for event in clean:
    missing = REQUIRED_FIELDS - set(event)   # required, minus what is present
    print(event["event_id"], missing)
    if missing:
        failed.append(event["event_id"])

print("failed schema:", failed)

### Question 5

Parsing the amount column. -> 7 tuples, with `None` for `e1`, `e4` and `e7`; the total prints `79.99`.

`.get()` rather than `[]` is load-bearing: `e7` has no `amount` key at all,
and the bracket form would have raised on it.

NOTE WHAT THAT 79.99 ACTUALLY IS. It is the sum of every amount, and a
refund is stored as a positive number — so it counts 12.50 leaving the
business as 12.50 arriving. It is a correct sum of the wrong thing, and
Q11 comes back to it.

In [ ]:
parsed = []
for event in clean:
    raw = event.get("amount")          # .get() -> None when the key is absent
    value = float(raw) if raw is not None else None
    parsed.append((event["event_id"], event["action"], value))

print(parsed)

print(round(sum([v for _id, _a, v in parsed if v is not None]), 2))

PART C — Joining to a dimension

### Question 6

Joining to the dimension. -> `e1 Ada East`, `e2 Bo West`, `e3 Ada East`, `e4 Cai East`, `e5 UNKNOWN UNKNOWN`, `e6 Ada East`, `e7 Bo West`.

A dictionary IS the lookup — `users[uid]` is the join, and it is roughly
instant however large the dimension grows. That is why lookups get loaded
into dicts rather than scanned as lists.

`.get(key, UNKNOWN)` makes it a LEFT join: the row survives and is marked.
`users[event["user"]]` would have raised on `e5` and stopped the pipeline —
which is the safer failure, because at least you would know.

In [ ]:
UNKNOWN = {"name": "UNKNOWN", "region": "UNKNOWN"}

for event in clean:
    profile = users.get(event["user"], UNKNOWN)   # left join, in one call
    print(event["event_id"], profile["name"], profile["region"])

### Question 7

The anti-join. -> events have `{u01, u02, u03, u99}`, the lookup has `{u01, u02, u03}`, orphans are `{u99}`, accounting for `['e5']` — 1 record.

One set difference answers "which facts have no matching dimension row" —
the `NOT EXISTS` / `LEFT JOIN ... IS NULL` pattern from week 2, day 2, in three
characters.

RUN THIS EVERY TIME YOU JOIN. Q6 already hid the problem behind a tidy
`UNKNOWN`, and Q11 shows what it costs: any region report loses u99's money
without a trace. The orphan count is the only thing standing between you
and a total that is quietly short.

In [ ]:
event_users = {event["user"] for event in clean}
known_users = set(users)
orphans = event_users - known_users        # in the facts, not the dimension

print("in events:", event_users)
print("in lookup:", known_users)
print("orphans:  ", orphans)

orphan_rows = [e["event_id"] for e in clean if e["user"] in orphans]
print("orphan records:", orphan_rows, len(orphan_rows))

PART D — Aggregating

### Question 8

Grouping and totalling. -> counts `{'view': 2, 'purchase': 4, 'refund': 1}`; totals `{'view': 0.0, 'purchase': 67.49, 'refund': 12.5}`; sorted, purchase leads on both.

Two accumulators filled by one pass. Building them together keeps them
consistent — separate loops drift apart the moment someone edits one.

Note the raw sorted output shows `67.49000000000001`: binary floating point,
not a bug. Round at the point of display, never in the accumulation.

And purchase is 4 records but only 67.49, because `e7` is a purchase with no
amount. Count and sum disagree, and both are right.

In [ ]:
by_action = {}
totals = {}
for _id, action, value in parsed:
    if action not in by_action:
        by_action[action] = 0
        totals[action] = 0.0
    by_action[action] = by_action[action] + 1
    totals[action] = totals[action] + (value if value is not None else 0.0)

print(by_action)
print({k: round(v, 2) for k, v in totals.items()})

print(sorted(by_action.items(), key=lambda pair: pair[1], reverse=True))
print(sorted(totals.items(), key=lambda pair: pair[1], reverse=True))

### Question 9

Distinct users per action. -> `{'view': {u01, u03}, 'purchase': {u99, u01, u02}, 'refund': {u01}}`, giving `{'view': 2, 'purchase': 3, 'refund': 1}` against record counts `{'view': 2, 'purchase': 4, 'refund': 1}`.

PURCHASE: 4 RECORDS, 3 USERS. The set collapses u02's two purchases into
one person, which is exactly what "how many customers bought" means.

Reporting 4 there would be the same grain error as `COUNT(*)` versus
`COUNT(DISTINCT CustomerID)` in week 2, day 2, and it inflates in the direction
that flatters you — which is why it survives review.

A dict-of-sets is the structure for this. Nothing else deduplicates for
free while still grouping.

In [ ]:
users_per_action = {}
for event in clean:
    action = event["action"]
    if action not in users_per_action:
        users_per_action[action] = set()
    users_per_action[action].add(event["user"])

print(users_per_action)
print({k: len(v) for k, v in users_per_action.items()})
print(by_action)   # records, for comparison -- purchase differs

### Question 10

Net revenue per user. -> `{'u01': 0.0, 'u02': 49.99, 'u03': 0.0, 'u99': 5.0}`, ranked `u02`, `u99`, then `u01` and `u03` on zero.

LOOK AT u01 AND u03. Both show 0.00 and they mean opposite things: u03
browsed and never bought, while u01 bought 12.50 and refunded 12.50. The
number cannot tell them apart, and any model treating "zero revenue" as
"not a customer" would file an active refunder alongside a window shopper.

And `u99` is second on the list despite not existing in your user table.

In [ ]:
net = {}
for event in clean:
    user = event["user"]
    raw = event.get("amount")
    value = float(raw) if raw is not None else 0.0
    if event["action"] == "refund":
        value = -value
    elif event["action"] != "purchase":
        value = 0.0
    if user not in net:
        net[user] = 0.0
    net[user] = net[user] + value

print({k: round(v, 2) for k, v in net.items()})
print(sorted(net.items(), key=lambda pair: pair[1], reverse=True))

PART E — Stretch: the data quality report

The cell nobody writes and everybody needs. It is also where the numbers on
this sheet stop agreeing with each other.

### Question 11

Stretch — the quality report. -> rows in `8`, after dedup `7`, dropped `1`, failing schema `['e7']`, orphans `{'u99'}`, null/missing amount `3`, and the three figures: sum of all amounts `79.99`, gross purchase value `67.49`, net revenue `54.99`.

THREE DEFENSIBLE REVENUE FIGURES, AND THEY DIFFER BY 45%.

`79.99` sums every amount — and refunds are stored positive, so it counts
money leaving as money arriving. It is also the one you get by writing the
obvious `sum()`, which is why it is the one that ends up on slides.
`67.49` is gross purchases. `54.99` is what the business kept. Only the
last is revenue.

Nothing raised. No row was lost. Every one of those numbers is an accurate
sum of something — the question is only ever WHICH something, and the code
does not record which one you meant.

The rest of the report exists so the reader can tell: 1 duplicate dropped,
1 record failing schema, 1 orphan user whose 5.00 vanishes from any regional
breakdown, 3 records with no amount. Ship the caveats WITH the number.
A figure with no quality report beside it is a claim, not a measurement.

In [ ]:
print("rows in:             ", len(raw_events))
print("rows after dedup:    ", len(clean))
print("duplicates dropped:  ", len(raw_events) - len(clean))
print("failing schema:      ", failed)
print("orphan users:        ", orphans)
print("null/missing amount: ", len([1 for _i, _a, v in parsed if v is None]))

all_amounts = sum([v for _i, _a, v in parsed if v is not None])
gross = totals["purchase"]
net_total = sum(net.values())
print("sum of all amounts:  ", round(all_amounts, 2))
print("gross purchase value:", round(gross, 2))
print("net revenue:         ", round(net_total, 2))

# WHICH NUMBERS MISLEAD, AND WHY:
#
# 0. THE HEADLINE: three defensible "revenue" figures, and they differ.
#    79.99 sums every amount -- and a refund is stored as a POSITIVE amount,
#    so this one counts money going out as money coming in. 67.49 is gross
#    purchases. 54.99 is what the business actually kept. Only the last is
#    revenue, and the first is the one you get by writing the obvious sum().
#
# 1. Any report grouped by REGION silently drops u99's 5.00, because u99 is
#    not in the lookup and has no region. The regional totals will still add
#    up and still look complete. Q7 is the only thing that reveals it.
#
# 2. "purchase: 4 records" and "purchase value 67.49" disagree, because e7
#    is a purchase with no amount. Counting records and summing money are
#    different questions, and this record answers them differently.
#
# 3. Net revenue for u01 is 0.00 -- a real 12.50 purchase and a real 12.50
#    refund. "Zero" means "cancelled out", not "did nothing", and a churn
#    model treating those the same would be wrong.
#
# 4. Distinct users per action (Q9) is smaller than record counts (Q8) for
#    purchase, because u01 and u02 each appear more than once. Reporting
#    record counts as "customers" is the most common inflation there is.